# Perturbation Prediction Benchmark — Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kagtgi/PerturbationBenchmarkTool/blob/main/eval/notebooks/tutorial.ipynb)

This notebook walks you through benchmarking five perturbation prediction models
on your own Perturb-seq `.h5ad` dataset — from loading data to a final comparison table.

---

## Supported models

| Model | Approach | GPU | ~Runtime | ~Disk |
|-------|----------|-----|----------|-------|
| **GEARS** | Graph neural network | Recommended | 10–20 min | 200 MB |
| **scGPT** | Pre-trained Transformer | Recommended | 20–30 min | 500 MB |
| **STATE** | Foundation model (HVG) | Recommended | 15–25 min | 1 GB |
| **Cell2Sentence** | LLM-based (Gemma-2-2B) | **Required** | 60–90 min | 10 GB |
| **CPA** | Conditional Perturbation Autoencoder | Recommended | 20–30 min | 300 MB |

**First run**: each model downloads pretrained weights automatically. All five together require ~12 GB of disk space and internet access.

## Evaluation tiers

| Tier | Metrics | Direction |
|------|---------|----------|
| T1 — Whole-profile identity | Centroid Accuracy, Profile Distance Score, Systema Pearson Delta | ↑ ↓ ↑ |
| T2 — DE gene accuracy | Directional Accuracy, Pearson Delta Top-K, Jaccard Top-K | ↑ ↑ ↑ |
| T3 — Distributional fidelity | Energy Distance, MMD (RBF) | ↓ ↓ |

---
## Step 1 — Setup

**Colab users**: select a **GPU runtime** before running (`Runtime → Change runtime type → T4 GPU`).
Cell2Sentence requires a GPU; all other models run faster with one.

Clone the repository and install base dependencies.
Model-specific packages install themselves automatically on first use.

In [ ]:
import os, subprocess, sys

# ── 1. Clone repo (Colab) ─────────────────────────────────────────────────
if not os.path.isdir('PerturbationBenchmarkTool'):
    subprocess.check_call(['git', 'clone', '--quiet',
                           'https://github.com/kagtgi/PerturbationBenchmarkTool.git'])
%cd PerturbationBenchmarkTool

# ── 2. Base dependencies ──────────────────────────────────────────────────
# torch is pre-installed in Colab; for local installs follow pytorch.org/get-started
!pip install -q anndata scanpy scipy pandas matplotlib

# ── 3. Verify working directory ───────────────────────────────────────────
assert os.path.isdir('eval'), (
    "Working directory must be the repo root (PerturbationBenchmarkTool). "
    "Run: %cd PerturbationBenchmarkTool"
)
print('Setup OK — working directory:', os.getcwd())

# ── 4. GPU check ──────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory // 1024**3} GB VRAM)')
else:
    print('WARNING: No GPU detected. Cell2Sentence will fail. All other models will run slowly on CPU.')

---
## Step 2 — Configure

Edit the four settings below to match your dataset.
Everything else (subsampling fraction, metric thresholds, timeouts) lives in `eval/config.py`
and can be changed there if needed.

In [ ]:
from eval import config
import torch

# ── EDIT THESE ─────────────────────────────────────────────────────────────
config.DATA_PATH   = 'K562.h5ad'        # path to your .h5ad Perturb-seq file
config.CTRL_LABEL  = 'non-targeting'    # obs value that identifies control cells
config.PERT_COL    = 'gene'             # obs column holding perturbation gene names
config.OUTPUT_DIR  = 'results'          # directory for saving outputs
# ───────────────────────────────────────────────────────────────────────────

# Auto-detect GPU; override with 'cpu' if needed
config.DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
config.RANDOM_SEED = 42

print(f'Data path : {config.DATA_PATH}')
print(f'Ctrl label: {config.CTRL_LABEL!r}  (in obs[{config.PERT_COL!r}])')
print(f'Device    : {config.DEVICE}')
print(f'Output    : {config.OUTPUT_DIR}')

---
## Step 3 — Inspect Dataset

Load the `.h5ad` file and verify the perturbation column and control label.
The benchmark runner subsamples the data internally, so this step is optional
but useful for confirming your configuration.

In [ ]:
import scanpy as sc

adata = sc.read_h5ad(config.DATA_PATH)
print(f'Shape         : {adata.shape[0]:,} cells × {adata.shape[1]:,} genes')

# Verify perturbation column exists
assert config.PERT_COL in adata.obs.columns, (
    f"Column '{config.PERT_COL}' not found in adata.obs. "
    f"Available: {list(adata.obs.columns)}"
)
all_perts = adata.obs[config.PERT_COL].unique()

# Verify control label exists
assert config.CTRL_LABEL in all_perts, (
    f"Control label '{config.CTRL_LABEL}' not found in obs['{config.PERT_COL}']. "
    f"Found: {list(all_perts[:10])}..."
)
n_perts = (all_perts != config.CTRL_LABEL).sum()
n_ctrl  = (adata.obs[config.PERT_COL] == config.CTRL_LABEL).sum()
print(f'Control cells : {n_ctrl:,}')
print(f'Perturbations : {n_perts} (unique gene knockouts)')
print('Config looks good!')

---
## Step 4 — Run All Models

`eval_runner.run()` handles everything:
- Subsamples the data **once** (all models see identical cells)
- Runs each model in a **separate subprocess** — no library conflicts, no runtime restart needed
- Enforces the safe run order: `gears → scgpt → cell2sentence → cpa → state`
- Streams live logs and prints any errors automatically at the end
- Saves one JSON result file per model in `OUTPUT_DIR`

**Expected total time**: 2–4 hours (with GPU), dominated by Cell2Sentence (~1–1.5 h).
To skip Cell2Sentence, remove it from the `models` list below.

In [ ]:
from eval.eval_runner import run

results = run(
    data_path  = config.DATA_PATH,
    models     = ['gears', 'scgpt', 'cell2sentence', 'cpa', 'state'],
    device     = config.DEVICE,
    seed       = config.RANDOM_SEED,
    output_dir = config.OUTPUT_DIR,
    # isolate=True,   # default — each model in its own subprocess (recommended)
)

# Summary
print()
print(f"{'Model':<20}  {'Status':<10}  {'Perts':>6}  {'Runtime':>10}")
print('-' * 55)
for r in results:
    rt  = f"{r.get('runtime_seconds', 0) / 60:.1f} min"
    print(f"{r['model']:<20}  {r.get('status','?'):<10}  "
          f"{len(r.get('pert_names', [])):>6}  {rt:>10}")

### Alternative: run a single model (in-process)

Use this when you want to test one model quickly without launching a subprocess.
**Note**: heavy models (Cell2Sentence, CPA) will block the notebook kernel for their full runtime.

In [ ]:
import json, os
from eval.dataset import load_h5ad, ensure_raw_counts
from eval.sampling import stratified_subsample
from eval.models import run_model_eval

# ── Choose one model ──────────────────────────────────────────────────────
MODEL_NAME = 'gears'   # options: gears | scgpt | state | cell2sentence | cpa
# ─────────────────────────────────────────────────────────────────────────

# Load and subsample
adata = load_h5ad(config.DATA_PATH)
adata = ensure_raw_counts(adata)
adata_sub = stratified_subsample(adata)
print(f'Subsampled: {adata_sub.shape[0]:,} cells, {adata_sub.obs[config.PERT_COL].nunique() - 1} perturbations')

# Run evaluation (installs model deps automatically on first call)
result = run_model_eval(MODEL_NAME, adata_sub.copy(), vars(config))

# Display
print(f"\nModel   : {result['model']}")
print(f"Perts   : {len(result['pert_names'])}")
print(f"Runtime : {result['runtime_seconds']:.1f} s")
print("\nMetrics:")
for k, v in result['metrics'].items():
    print(f"  {k}: {v:.4f}")

# Save
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
out = f"{config.OUTPUT_DIR}/{MODEL_NAME}_results.json"
with open(out, 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\nSaved: {out}")

---
## Step 5 — Compare Results

Aggregate all `*_results.json` files from `OUTPUT_DIR` into a single comparison table.

In [ ]:
from eval.collect_results import collect, pretty_table

df = collect(config.OUTPUT_DIR)

if df.empty:
    print('No results found yet. Run Step 4 first.')
else:
    print(pretty_table(df))

In [ ]:
# View as a styled DataFrame
metric_cols = [c for c in df.columns if c.startswith('T')]
df[['model'] + metric_cols]

In [ ]:
# Export results
import os
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

csv_path = f'{config.OUTPUT_DIR}/final_comparison.csv'
df.to_csv(csv_path, index=False)
print(f'Saved: {csv_path}')

---
## Step 6 — Interpret Results

### Tier 1 — Whole-Profile Identity
- **Centroid Accuracy (CA)** ↑ — Is the predicted mean profile closest to the correct perturbation's mean? (1.0 = always correct)
- **Profile Distance Score (PDS)** ↓ — Relative rank of the true perturbation by profile distance. (0.0 = always ranked #1)
- **Systema Pearson Delta** ↑ — Genome-wide Pearson correlation of predicted vs observed log-fold-changes.

### Tier 2 — Differentially Expressed Gene Accuracy
- **Directional Accuracy** ↑ — Fraction of top DE genes where the predicted direction (up/down) is correct.
- **Pearson Delta Top-K** ↑ — Pearson correlation restricted to the top-K DE genes.
- **Jaccard DE Top-K** ↑ — Overlap (Jaccard index) between predicted and observed top-K DE gene sets.

### Tier 3 — Distributional Fidelity
- **Energy Distance** ↓ — Energy distance between the distributions of predicted and observed cells.
- **MMD (RBF)** ↓ — Maximum Mean Discrepancy (RBF kernel) between the two distributions.

**`K` = `TOP_K_DE` (default 50).** Change it in `eval/config.py`.

---
## Troubleshooting

Errors and tracebacks are printed automatically after `run()` finishes.
Use the helpers below for deeper inspection.

In [ ]:
from eval.eval_runner import print_errors, print_logs

# Show error message + traceback for every failed model:
print_errors(config.OUTPUT_DIR)

# Show full subprocess log (stdout + stderr) for a specific model:
# print_logs(config.OUTPUT_DIR, model='gears')

# Show logs for all models:
# print_logs(config.OUTPUT_DIR)

# Inspect raw result JSON directly:
# import json
# print(json.dumps(json.load(open(f'{config.OUTPUT_DIR}/gears_results.json')), indent=2))

### Common issues

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| All models fail in < 5 s | Wrong working directory | The notebook must run from `PerturbationBenchmarkTool/` — rerun the `%cd` in Step 1 |
| `ModuleNotFoundError: No module named 'eval'` | Wrong working directory | See above |
| `CUDA not available` / Cell2Sentence fails | GPU required for C2S | Enable a GPU runtime (`Runtime → Change runtime type → T4 GPU`) |
| `Gene overlap = 0` (CPA) | Control label mismatch | Set `config.CTRL_LABEL` to the exact value used in your dataset's perturbation column |
| `KeyError: 'gene'` | Wrong `PERT_COL` | Set `config.PERT_COL` to the correct column name — run Step 3 to check available columns |
| Model times out | Dataset too large or slow hardware | Increase model timeout: pass `timeout=120` to `run()` or lower `config.SUBSAMPLE_FRAC` |
| CPA fails with git error | `git` not installed | `!apt install -y git` (Colab) or `brew install git` / `apt install git` (local) |
| STATE fails after install | `uv` path issue | `!pip install uv` then rerun; or restart the Colab runtime once |